# Fine-Tune Transformer Models on New Datasets

Trains **CodeBERT**, **GraphCodeBERT**, and **UniXcoder** on:
- `llmgen` — OSS-forge/HumanVsAICode (Python + Java, ChatGPT/DeepSeek/Qwen)
- `csn`    — basakdemirok/AIGCodeSet  (Python only, CodeLlama/Codestral/Gemini)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your project zip to `/content/` (see Cell 2)
3. Mount Google Drive so models are saved persistently (Cell 3)

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch runtime to T4 GPU!')

In [ ]:
# ── Cell 2: Upload & extract project zip ─────────────────────────────────────
# Option A — upload from your machine
from google.colab import files
uploaded = files.upload()          # select your project zip
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

import zipfile, os
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content')
print('Extracted to /content')

# Option B — uncomment if the zip is already in Drive
# !cp "/content/drive/MyDrive/AI-GCD/project.zip" /content/
# !unzip -q /content/project.zip -d /content

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Where trained models will be saved on Drive
DRIVE_MODELS = '/content/drive/MyDrive/AI-GCD/models'
import os
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f'Models will be saved to: {DRIVE_MODELS}')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 datasets accelerate loguru optuna xgboost \
              scikit-learn pandas pyarrow safetensors

In [ ]:
# ── Cell 5: Python path & imports ────────────────────────────────────────────
import sys
sys.path.insert(0, '/content')

import os, math, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 6: Configuration ─────────────────────────────────────────────────────
# Datasets to train on
DATASETS = {
    'llmgen': '/content/data/splits_llmgen',   # OSS-forge/HumanVsAICode
    'csn':    '/content/data/splits_csn',      # basakdemirok/AIGCodeSet
}

# Models to fine-tune
MODELS = {
    'codebert':      'microsoft/codebert-base',
    'graphcodebert': 'microsoft/graphcodebert-base',
    'unixcoder':     'microsoft/unixcoder-base',
}

# Training hyperparameters
MAX_LENGTH   = 512
BATCH_SIZE   = 16       # reduce to 8 if you get OOM errors
EPOCHS       = 5
LR           = 2e-5
WARMUP_RATIO = 0.10
SAVE_STEPS   = 250
PATIENCE     = 6        # early stopping patience (in evals)

# Where to save locally (will be synced to Drive after each model)
LOCAL_MODELS = '/content/models'

# Verify splits exist
for ds_key, splits_path in DATASETS.items():
    p = Path(splits_path)
    if p.exists():
        train_rows = len(pd.read_parquet(p / 'train.parquet'))
        test_rows  = len(pd.read_parquet(p / 'test.parquet'))
        print(f'✓ {ds_key}: train={train_rows}, test={test_rows}')
    else:
        print(f'✗ {ds_key}: NOT FOUND at {splits_path}')
        print(f'  → Upload splits_llmgen / splits_csn folders to /content/data/')

In [ ]:
# ── Cell 7: Upload dataset splits (if not already in /content/data/) ─────────
# Run this only if splits are missing above.
# Option A — upload zip of splits from your machine:
# from google.colab import files
# upl = files.upload()   # upload splits_llmgen.zip and/or splits_csn.zip
# import zipfile
# for name in upl:
#     with zipfile.ZipFile(name) as zf:
#         zf.extractall('/content/data')
#     print(f'Extracted {name}')

# Option B — copy from Drive:
# !cp -r "/content/drive/MyDrive/AI-GCD/splits_llmgen" /content/data/
# !cp -r "/content/drive/MyDrive/AI-GCD/splits_csn"    /content/data/

print('Skip this cell if splits already exist above.')

In [ ]:
# ── Cell 8: Core model + dataset classes ──────────────────────────────────────
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer

class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.codes  = df['code'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):  return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float),
        }


class CodeBERTClassifier(nn.Module):
    def __init__(self, model_name, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        encoder_dim     = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(encoder_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, input_ids, attention_mask=None, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls    = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls).squeeze(-1)
        loss   = None
        if labels is not None:
            loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        return {'loss': loss, 'logits': logits}

print('Classes defined.')

In [ ]:
# ── Cell 9: Metrics + Trainer ─────────────────────────────────────────────────
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = 1 / (1 + np.exp(-logits))
    preds  = (probs >= 0.5).astype(int)
    labels = labels.astype(int)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, zero_division=0)),
        'auc':      float(roc_auc_score(labels, probs)) if len(set(labels)) > 1 else 0.0,
    }


class CodeBERTTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs, labels=labels)
        loss    = outputs['loss']
        return (loss, outputs) if return_outputs else loss


def clean_corrupt_checkpoints(ckpt_dir: Path):
    """Remove checkpoints that are missing trainer_state.json (from interrupted runs)."""
    for ckpt in sorted(ckpt_dir.glob('checkpoint-*')):
        if not (ckpt / 'trainer_state.json').exists():
            print(f'Removing corrupt checkpoint: {ckpt}')
            shutil.rmtree(ckpt)

print('Trainer helpers defined.')

In [ ]:
# ── Cell 10: Training function ────────────────────────────────────────────────
from transformers.trainer_utils import get_last_checkpoint

def train_model(model_key, hf_model_name, splits_dir, ds_key):
    """
    Fine-tune one transformer model on one dataset.

    model_key     : 'codebert' | 'graphcodebert' | 'unixcoder'
    hf_model_name : HuggingFace model ID
    splits_dir    : path to folder with train/val/test.parquet
    ds_key        : 'llmgen' | 'csn'
    """
    splits_dir  = Path(splits_dir)
    output_dir  = Path(LOCAL_MODELS) / ds_key / model_key
    final_dir   = Path(LOCAL_MODELS) / ds_key / f'{model_key}_final'
    drive_final = Path(DRIVE_MODELS) / ds_key / f'{model_key}_final'

    output_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'  Model   : {model_key} ({hf_model_name})')
    print(f'  Dataset : {ds_key} ({splits_dir})')
    print(f'  Output  : {output_dir}')
    print(f'{"="*60}')

    # Skip if final model already saved to Drive
    if drive_final.exists():
        print(f'  Already trained — found {drive_final}. Skipping.')
        return

    # Load data
    train_df = pd.read_parquet(splits_dir / 'train.parquet')
    val_df   = pd.read_parquet(splits_dir / 'val.parquet')
    test_df  = pd.read_parquet(splits_dir / 'test.parquet')
    print(f'  Rows — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}')

    # Tokenise
    tokenizer     = AutoTokenizer.from_pretrained(hf_model_name)
    train_dataset = CodeDataset(train_df, tokenizer, MAX_LENGTH)
    val_dataset   = CodeDataset(val_df,   tokenizer, MAX_LENGTH)
    test_dataset  = CodeDataset(test_df,  tokenizer, MAX_LENGTH)

    # Build model
    model = CodeBERTClassifier(hf_model_name)

    # Checkpoint handling — resume if interrupted
    clean_corrupt_checkpoints(output_dir)
    resume_from = get_last_checkpoint(str(output_dir))
    if resume_from:
        print(f'  Resuming from: {resume_from}')

    steps_per_epoch = max(1, math.ceil(len(train_dataset) / BATCH_SIZE))
    save_steps      = min(SAVE_STEPS, steps_per_epoch)
    print(f'  Batches/epoch: {steps_per_epoch}  |  Save every {save_steps} steps')

    args = TrainingArguments(
        output_dir                  = str(output_dir),
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        learning_rate               = LR,
        warmup_ratio                = WARMUP_RATIO,
        weight_decay                = 0.01,
        evaluation_strategy         = 'steps',
        save_strategy               = 'steps',
        eval_steps                  = save_steps,
        save_steps                  = save_steps,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        greater_is_better           = True,
        logging_steps               = min(50, save_steps),
        fp16                        = torch.cuda.is_available(),
        dataloader_num_workers      = 2,
        report_to                   = 'none',
    )

    trainer = CodeBERTTrainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    trainer.train(resume_from_checkpoint=resume_from)

    # ── Save final model locally
    final_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f'  Final model saved locally: {final_dir}')

    # ── Evaluate on test set
    test_results = trainer.predict(test_dataset)
    logits = test_results.predictions
    labels = test_results.label_ids.astype(int)
    probs  = 1 / (1 + np.exp(-logits))
    preds  = (probs >= 0.5).astype(int)

    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    metrics = {
        'dataset':  ds_key,
        'model':    model_key,
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, zero_division=0)),
        'auc':      float(roc_auc_score(labels, probs)),
    }
    print(f'  TEST → AUC={metrics["auc"]:.4f}  F1={metrics["f1"]:.4f}  Acc={metrics["accuracy"]:.4f}')

    # Save metrics alongside the model
    (final_dir / 'test_metrics.json').write_text(json.dumps(metrics, indent=2))

    # ── Sync to Google Drive
    drive_final.parent.mkdir(parents=True, exist_ok=True)
    if drive_final.exists():
        shutil.rmtree(drive_final)
    shutil.copytree(str(final_dir), str(drive_final))
    print(f'  Synced to Drive: {drive_final}')

    return metrics

print('train_model() defined.')

---
## Run Training

Choose which dataset and model to train. Each cell trains one model on one dataset.
Results are saved to Drive automatically after each model completes.

In [ ]:
# ── Cell 11: Train ALL models on llmgen ──────────────────────────────────────
# OSS-forge/HumanVsAICode — ~28k train rows, Python + Java
# Expected time: ~45 min per model on T4

llmgen_results = []
for model_key, hf_name in MODELS.items():
    r = train_model(model_key, hf_name, DATASETS['llmgen'], 'llmgen')
    if r:
        llmgen_results.append(r)

print('\n=== llmgen summary ===')
for r in llmgen_results:
    print(f"  {r['model']:15s}  AUC={r['auc']:.4f}  F1={r['f1']:.4f}  Acc={r['accuracy']:.4f}")

In [ ]:
# ── Cell 12: Train ALL models on csn ─────────────────────────────────────────
# basakdemirok/AIGCodeSet — ~10k train rows, Python only
# Expected time: ~30 min per model on T4

csn_results = []
for model_key, hf_name in MODELS.items():
    r = train_model(model_key, hf_name, DATASETS['csn'], 'csn')
    if r:
        csn_results.append(r)

print('\n=== csn summary ===')
for r in csn_results:
    print(f"  {r['model']:15s}  AUC={r['auc']:.4f}  F1={r['f1']:.4f}  Acc={r['accuracy']:.4f}")

In [ ]:
# ── Cell 13: Train a single model (one at a time) ────────────────────────────
# Use this if you prefer to train one model per session to avoid timeouts.

# Change these two variables:
TRAIN_MODEL   = 'codebert'    # 'codebert' | 'graphcodebert' | 'unixcoder'
TRAIN_DATASET = 'llmgen'      # 'llmgen' | 'csn'

r = train_model(
    TRAIN_MODEL,
    MODELS[TRAIN_MODEL],
    DATASETS[TRAIN_DATASET],
    TRAIN_DATASET,
)
if r:
    print(f"\nResult: AUC={r['auc']:.4f}  F1={r['f1']:.4f}  Acc={r['accuracy']:.4f}")

---
## After Training: Update cross_dataset_results.json

Run Cell 14 after all models have finished to merge the new transformer results
into the existing results file.

In [ ]:
# ── Cell 14: Collect all results and update results JSON ──────────────────────
results_path = Path('/content/results/cross_dataset_results.json')
results_path.parent.mkdir(exist_ok=True)

# Load existing (classical ML) results if present
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f'Loaded existing results: {list(results.keys())}')
else:
    results = {}

# Scan Drive for test_metrics.json in each model_final folder
for ds_key in ['llmgen', 'csn']:
    if ds_key not in results:
        results[ds_key] = {}
    for model_key in MODELS:
        metrics_file = Path(DRIVE_MODELS) / ds_key / f'{model_key}_final' / 'test_metrics.json'
        if metrics_file.exists():
            m = json.loads(metrics_file.read_text())
            results[ds_key][model_key] = {
                'accuracy': m['accuracy'],
                'f1':       m['f1'],
                'auc':      m['auc'],
            }
            print(f'  {ds_key}/{model_key}: AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}')
        else:
            print(f'  {ds_key}/{model_key}: not found yet')

results_path.write_text(json.dumps(results, indent=2))
print(f'\nSaved to {results_path}')

# Also save to Drive
drive_results = Path(DRIVE_MODELS).parent / 'cross_dataset_results.json'
drive_results.write_text(json.dumps(results, indent=2))
print(f'Saved to Drive: {drive_results}')

In [ ]:
# ── Cell 15: Print final results table ────────────────────────────────────────
results_path = Path('/content/results/cross_dataset_results.json')
if not results_path.exists():
    results_path = Path(DRIVE_MODELS).parent / 'cross_dataset_results.json'

results = json.loads(results_path.read_text())

all_models = ['xgboost', 'random_forest', 'logistic_regression',
              'codebert', 'graphcodebert', 'unixcoder']
datasets   = ['codenet', 'llmgen', 'csn']

print(f'{'Model':<22}  {'CodeNet AUC':>12}  {'llmgen AUC':>12}  {'csn AUC':>12}')
print('-' * 65)
for m in all_models:
    row = []
    for ds in datasets:
        v = results.get(ds, {}).get(m)
        row.append(f"{v['auc']*100:.2f} %" if v else '  pending ')
    print(f'{m:<22}  {row[0]:>12}  {row[1]:>12}  {row[2]:>12}')

In [ ]:
# ── Cell 16: Download models back to your machine (optional) ──────────────────
# Run this if you want to use the trained models locally.
# Creates a zip of all new dataset models and downloads it.

import zipfile
from google.colab import files

zip_path = '/content/new_dataset_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ds_key in ['llmgen', 'csn']:
        for model_key in MODELS:
            model_dir = Path(DRIVE_MODELS) / ds_key / f'{model_key}_final'
            if model_dir.exists():
                for f in model_dir.rglob('*'):
                    if f.is_file():
                        zf.write(f, f.relative_to(Path(DRIVE_MODELS).parent))
                print(f'  Added {ds_key}/{model_key}_final')

print(f'\nZip size: {Path(zip_path).stat().st_size / 1e6:.0f} MB')
files.download(zip_path)